# Notebook 01 — ETL: Bronze → Silver

## Arquitectura

```
data/raw/  ──► S3 Bronze  ──► S3 Silver
(CSVs)        (datos         (datos limpios
               crudos)        y validados)
```

| Capa   | Contenido                                      |
|--------|------------------------------------------------|
| Bronze | CSVs originales de Kaggle sin modificar        |
| Silver | Datos limpios, tipos correctos, sin duplicados |

In [1]:
%pip install -r ../requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


## 0. Configuración

In [ ]:
import logging
import yaml
import boto3
import pandas as pd
import awswrangler as wr
from pathlib import Path
from datetime import datetime

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

# Cargar config
with open("../config.yaml") as f:
    CFG = yaml.safe_load(f)

BUCKET       = CFG["aws"]["s3_bucket"]
BRONZE_PATH  = f"s3://{BUCKET}/{CFG['aws']['s3_paths']['bronze']}"
SILVER_PATH  = f"s3://{BUCKET}/{CFG['aws']['s3_paths']['silver']}"

log.info(f"Bucket : {BUCKET}")
log.info(f"Bronze : {BRONZE_PATH}")
log.info(f"Silver : {SILVER_PATH}")

2026-05-22 16:49:18,356 [INFO] Bucket : itam-final-azucena


2026-05-22 16:49:18,356 [INFO] Bronze : s3://itam-final-azucena/auto-repair-shop/bronze


2026-05-22 16:49:18,357 [INFO] Silver : s3://itam-final-azucena/auto-repair-shop/silver


## 1. Carga de CSVs locales

In [ ]:
RAW_LOGISTICS = Path("../") / CFG["etl"]["raw_files"]["logistics"]
RAW_SERVICE   = Path("../") / CFG["etl"]["raw_files"]["service"]

log.info(f"Leyendo: {RAW_LOGISTICS}")
df_logistics = pd.read_csv(RAW_LOGISTICS)

log.info(f"Leyendo: {RAW_SERVICE}")
df_service = pd.read_csv(RAW_SERVICE)

log.info(f"Logistics → {df_logistics.shape[0]:,} filas x {df_logistics.shape[1]} columnas")
log.info(f"Service   → {df_service.shape[0]:,} filas x {df_service.shape[1]} columnas")

2026-05-22 16:49:21,215 [INFO] Leyendo: ../data/raw/logistics_dataset_with_maintenance_required.csv


2026-05-22 16:49:21,233 [INFO] Leyendo: ../data/raw/services_and_repair.csv


2026-05-22 16:49:21,236 [INFO] Logistics → 5,000 filas x 23 columnas


2026-05-22 16:49:21,237 [INFO] Service   → 508 filas x 7 columnas


In [ ]:
# Vista rápida
df_logistics.head(3)

,Vehicle_ID,Make_and_Model,Year_of_Manufacture,Vehicle_Type,Usage_Hours,Route_Info,Load_Capacity,Actual_Load,Last_Maintenance_Date,Maintenance_Type,...,Vibration_Levels,Oil_Quality,Brake_Condition,Failure_History,Anomalies_Detected,Maintenance_Required,Weather_Conditions,Road_Conditions,Delivery_Times,center_id
0,1,Ford F-150,2009,SUV,18798.0,Highway,8000.0,6284.0,2025-06-23,Corrective,...,7.30,Fair,Worn,1,1,1,Rainy,Fair,6.7,5
1,2,Isuzu NPR,2019,Van,13530.0,Highway,20000.0,10339.0,2026-03-04,Preventive,...,2.27,Good,Fair,0,0,0,Rainy,Poor,9.3,1
2,3,Toyota Hilux,2005,Truck,18384.0,Urban,8000.0,6592.0,2025-03-04,Corrective,...,5.25,Poor,Fair,1,1,1,Foggy,Good,8.4,3


In [ ]:
df_service.head(3)

,CUSTOMER ID,CITY,STATE,SERVICE HISTORY,COMMON PROBLEM,SOLUTION USED,VEHICAL COMPANY
0,1,Ananthapuram,Andhra Pradesh,Oil Change; Brake Repair,Brake noise,Brake pad replacement,Maruti Suzuki
1,2,Ananthapuram,Andhra Pradesh,Engine Repair; Tire Rotation,Engine overheating,Radiator replacement,Hyundai
2,3,Ananthapuram,Andhra Pradesh,Bodywork; Paint Job,Dents and scratches,Body repair and repaint,Tata Motors


## 2. Subir a S3 Bronze (datos crudos sin modificar)

In [ ]:
log.info("Subiendo logistics a Bronze...")
wr.s3.to_parquet(
    df=df_logistics,
    path=f"{BRONZE_PATH}/logistics/",
    dataset=True,
    mode="overwrite"
)

log.info("Subiendo service a Bronze...")
wr.s3.to_parquet(
    df=df_service,
    path=f"{BRONZE_PATH}/service/",
    dataset=True,
    mode="overwrite"
)

log.info("Bronze OK")

2026-05-22 16:49:45,173 [INFO] Subiendo logistics a Bronze...


2026-05-22 16:49:45,254 [INFO] Initializing a Ray instance


2026-05-22 16:49:47,277	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 1909432320 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.55gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-05-22 16:49:48,410	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


2026-05-22 16:49:50,793 [INFO] Subiendo service a Bronze...


2026-05-22 16:49:51,354 [INFO] Bronze OK


## 3. Validación de columnas

Verificamos que los CSVs llegaron con todas las columnas esperadas según `config.yaml`.

In [ ]:
def validar_columnas(df: pd.DataFrame, nombre: str, esperadas: list) -> None:
    """
    Verifica que el DataFrame contenga todas las columnas esperadas.

    Args:
        df: DataFrame a validar.
        nombre: Nombre del dataset (para el log).
        esperadas: Lista de columnas requeridas desde config.yaml.

    Raises:
        ValueError: Si faltan columnas en el DataFrame.
    """
    faltantes = set(esperadas) - set(df.columns)
    if faltantes:
        raise ValueError(f"[{nombre}] Columnas faltantes: {faltantes}")
    log.info(f"[{nombre}] Columnas OK — {len(df.columns)} encontradas")


validar_columnas(df_logistics, "logistics", CFG["etl"]["expected_columns"]["logistics"])
validar_columnas(df_service,   "service",   CFG["etl"]["expected_columns"]["service"])

2026-05-22 16:49:57,502 [INFO] [logistics] Columnas OK — 23 encontradas


2026-05-22 16:49:57,503 [INFO] [service] Columnas OK — 7 encontradas


## 4. Limpieza - Logistics Dataset

In [ ]:
def limpiar_logistics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpieza y normalización del dataset de mantenimiento de flotilla.

    Operaciones:
        - Elimina duplicados por Vehicle_ID
        - Convierte Last_Maintenance_Date a datetime
        - Normaliza strings en columnas categóricas (strip + title case)
        - Castea columnas numéricas al tipo correcto
        - Elimina filas con nulos en columnas críticas

    Args:
        df: DataFrame crudo de logistics.

    Returns:
        DataFrame limpio listo para Silver.
    """
    original = len(df)
    log.info(f"Limpiando logistics ({original:,} filas)...")

    # Duplicados exactos
    df = df.drop_duplicates()
    log.info(f"  Duplicados exactos eliminados: {original - len(df)}")

    # Fechas
    df["Last_Maintenance_Date"] = pd.to_datetime(df["Last_Maintenance_Date"], errors="coerce")

    # Strings — quitar espacios y normalizar
    str_cols = ["Make_and_Model", "Vehicle_Type", "Route_Info",
                "Maintenance_Type", "Oil_Quality", "Brake_Condition",
                "Weather_Conditions", "Road_Conditions"]
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip().str.title()

    # Nulos en columnas principales
    criticas = ["Vehicle_ID", "Vehicle_Type", "Usage_Hours",
                "Maintenance_Required", "Last_Maintenance_Date"]
    antes = len(df)
    df = df.dropna(subset=criticas)
    log.info(f"  Filas con nulos en columnas críticas eliminadas: {antes - len(df)}")

    # Tipos numéricos
    num_cols = ["Usage_Hours", "Tire_Pressure",
            "Fuel_Consumption", "Battery_Status", "Vibration_Levels",
            "Failure_History", "Anomalies_Detected",
            "Load_Capacity", "Actual_Load", "Delivery_Times"]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Target como entero
    df["Maintenance_Required"] = df["Maintenance_Required"].astype(int)

    log.info(f"  Filas finales: {len(df):,}")
    return df


df_logistics_silver = limpiar_logistics(df_logistics)
df_logistics_silver.dtypes

2026-05-22 16:50:02,185 [INFO] Limpiando logistics (5,000 filas)...


2026-05-22 16:50:02,193 [INFO]   Duplicados exactos eliminados: 0


2026-05-22 16:50:02,221 [INFO]   Filas con nulos en columnas críticas eliminadas: 0


2026-05-22 16:50:02,224 [INFO]   Filas finales: 5,000


Vehicle_ID                        int64
Make_and_Model                   object
Year_of_Manufacture               int64
Vehicle_Type                     object
Usage_Hours                     float64
Route_Info                       object
Load_Capacity                   float64
Actual_Load                     float64
Last_Maintenance_Date    datetime64[ns]
Maintenance_Type                 object
Tire_Pressure                   float64
Fuel_Consumption                float64
Battery_Status                  float64
Vibration_Levels                float64
Oil_Quality                      object
Brake_Condition                  object
Failure_History                   int64
Anomalies_Detected                int64
Maintenance_Required              int64
Weather_Conditions               object
Road_Conditions                  object
Delivery_Times                  float64
center_id                         int64
dtype: object

## 5. Validación de rangos - Logistics

In [ ]:
def validar_rangos(df: pd.DataFrame, validaciones: dict) -> None:
    """
    Verifica que los valores numéricos estén dentro de rangos aceptables.

    Args:
        df: DataFrame a validar.
        validaciones: Diccionario con claves <col>_min / <col>_max desde config.yaml.

    Raises:
        Warning en log si hay valores fuera de rango (no detiene el pipeline).
    """
    checks = {
        "Usage_Hours":       ("usage_hours_min",    "usage_hours_max"),
        "Tire_Pressure":     ("tire_pressure_min",   "tire_pressure_max"),
    }
    for col, (k_min, k_max) in checks.items():
        v_min = validaciones[k_min]
        v_max = validaciones[k_max]
        fuera = df[(df[col] < v_min) | (df[col] > v_max)]
        if len(fuera) > 0:
            log.warning(f"  [{col}] {len(fuera)} filas fuera de rango [{v_min}, {v_max}]")
        else:
            log.info(f"  [{col}] Rango OK [{v_min}, {v_max}]")


validar_rangos(df_logistics_silver, CFG["etl"]["validation"])

2026-05-22 16:50:05,833 [INFO]   [Usage_Hours] Rango OK [0, 50000]


2026-05-22 16:50:05,835 [INFO]   [Tire_Pressure] Rango OK [10.0, 100.0]


## 6. Limpieza - Service Dataset

In [ ]:
def limpiar_service(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpieza y normalización del catálogo de problemas y soluciones.

    Operaciones:
        - Elimina duplicados
        - Strip de espacios en todas las columnas de texto
        - Normaliza nombre de compañía (quita espacios al inicio)
        - Elimina filas sin COMMON PROBLEM o SOLUTION USED

    Args:
        df: DataFrame crudo de service.

    Returns:
        DataFrame limpio listo para Silver.
    """
    log.info(f"Limpiando service ({len(df):,} filas)...")

    df = df.drop_duplicates()

    # eliminar espacios en blanco
    str_cols = ["SERVICE HISTORY", "COMMON PROBLEM", "SOLUTION USED", "VEHICAL COMPANY"]
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()

    # eliminar filas sin problema o solución
    antes = len(df)
    df = df.dropna(subset=["COMMON PROBLEM", "SOLUTION USED"])
    df = df[df["COMMON PROBLEM"] != "nan"]
    log.info(f"  Filas eliminadas por nulos: {antes - len(df)}")

    # Renombrar columnas a minúsculas
    df = df.rename(columns={
        "CUSTOMER ID":    "customer_id",
        "SERVICE HISTORY":"service_history",
        "COMMON PROBLEM": "common_problem",
        "SOLUTION USED":  "solution_used",
        "VEHICAL COMPANY":"vehicle_company",
        "CITY":           "city",
        "STATE":          "state"
    })

    log.info(f"  Filas finales: {len(df):,}")
    return df


df_service_silver = limpiar_service(df_service)
df_service_silver.head(3)

2026-05-22 16:50:09,907 [INFO] Limpiando service (508 filas)...


/tmp/ipykernel_580/361528120.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(str).str.strip()
/tmp/ipykernel_580/361528120.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(str).str.strip()
/tmp/ipykernel_580/361528120.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/sta

2026-05-22 16:50:09,915 [INFO]   Filas finales: 500


,customer_id,city,state,service_history,common_problem,solution_used,vehicle_company
0,1,Ananthapuram,Andhra Pradesh,Oil Change; Brake Repair,Brake noise,Brake pad replacement,Maruti Suzuki
1,2,Ananthapuram,Andhra Pradesh,Engine Repair; Tire Rotation,Engine overheating,Radiator replacement,Hyundai
2,3,Ananthapuram,Andhra Pradesh,Bodywork; Paint Job,Dents and scratches,Body repair and repaint,Tata Motors


## 7. Estadísticas de Silver

In [ ]:
print("=== LOGISTICS SILVER ===")
print(f"Filas: {len(df_logistics_silver):,}")
print(f"Nulos:\n{df_logistics_silver.isnull().sum()[df_logistics_silver.isnull().sum() > 0]}")
print(f"\nDistribución target:")
print(df_logistics_silver["Maintenance_Required"].value_counts(normalize=True).round(3))
print(f"\nVehicle_Type:")
print(df_logistics_silver["Vehicle_Type"].value_counts())

=== LOGISTICS SILVER ===
Filas: 5,000
Nulos:
Series([], dtype: int64)

Distribución target:
Maintenance_Required
1    0.696
0    0.304
Name: proportion, dtype: float64

Vehicle_Type:
Vehicle_Type
Truck    2002
Van      1508
Suv      1000
Bus       490
Name: count, dtype: int64


In [ ]:
print("=== SERVICE SILVER ===")
print(f"Filas: {len(df_service_silver):,}")
print(f"\nTop problemas:")
print(df_service_silver["common_problem"].value_counts().head(10))

=== SERVICE SILVER ===
Filas: 500

Top problemas:
common_problem
Cloudy headlights     49
Brake noise           45
Exhaust noise         37
Tire wear             32
Fluid leak            30
Engine overheating    22
Steering issues       22
Misaligned tires      19
Brake fluid leak      17
Dead battery          14
Name: count, dtype: int64


## 8. Guardar en S3 Silver

In [ ]:
log.info("Guardando logistics en Silver...")
wr.s3.to_parquet(
    df=df_logistics_silver,
    path=f"{SILVER_PATH}/logistics/",
    dataset=True,
    mode="overwrite"
)

log.info("Guardando service en Silver...")
wr.s3.to_parquet(
    df=df_service_silver,
    path=f"{SILVER_PATH}/service/",
    dataset=True,
    mode="overwrite"
)

log.info("Silver OK , ETL completo")

2026-05-22 16:50:25,017 [INFO] Guardando logistics en Silver...


2026-05-22 16:50:25,864 [INFO] Guardando service en Silver...


2026-05-22 16:50:26,450 [INFO] Silver OK , ETL completo


## 9. Verificación final

In [ ]:
# Leer de vuelta desde S3 para confirmar que se guardó correctamente
df_check_log = wr.s3.read_parquet(path=f"{SILVER_PATH}/logistics/")
df_check_svc = wr.s3.read_parquet(path=f"{SILVER_PATH}/service/")

log.info(f"Silver logistics leído desde S3: {df_check_log.shape}")
log.info(f"Silver service leído desde S3:   {df_check_svc.shape}")

assert len(df_check_log) == len(df_logistics_silver), "Discrepancia en logistics silver"
assert len(df_check_svc) == len(df_service_silver),   "Discrepancia en service silver"

log.info("Verificación OK")

2026-05-22 16:50:29,817 [INFO] Silver logistics leído desde S3: (5000, 23)


2026-05-22 16:50:29,818 [INFO] Silver service leído desde S3:   (500, 7)


2026-05-22 16:50:29,818 [INFO] Verificación OK
